# Compléments CNN Patch — Valeurs manquantes pour le mémoire

Ce notebook produit :
1. **K-means LAB sur le test set ISBI 2016** (379 images) → comparaison équitable avec le CNN
2. **Précision/Rappel pour P=32, 48, 64** → compléter le tableau d'ablation
3. **Post-traitement pour P=32** → remplacer le tableau actuel (qui est sur P=48)

**Pré-requis** : avoir exécuté `cnn_patch_segmentation.ipynb` (modèles sauvegardés dans `models/`).

In [2]:
import numpy as np
import os
import time
import json
from pathlib import Path
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedShuffleSplit
from skimage import color
from scipy.ndimage import label as ndimage_label
import tensorflow as tf
import pandas as pd

# ── Chemins (identiques à cnn_patch_segmentation.ipynb) ──
BASE = Path('.').resolve()
IMAGES_DIR    = BASE / 'dataset_ISIC' / 'ISBI2016_ISIC_Part1_Training_Data'
MASKS_DIR     = BASE / 'dataset_ISIC' / 'ISBI2016_ISIC_Part1_Training_GroundTruth'
TEST_IMG_DIR  = BASE / 'dataset_ISIC' / 'ISBI2016_ISIC_Part1_Test_Data'
TEST_MASK_DIR = BASE / 'dataset_ISIC' / 'ISBI2016_ISIC_Part1_Test_GroundTruth'
SAVE_DIR      = BASE / 'models'

IMG_SIZE = 256
SEED = 42

print(f'Test images : {len(list(TEST_IMG_DIR.glob("*.jpg")))}')
print(f'Modèles disponibles : {list(SAVE_DIR.glob("patch_cnn_p*.keras"))}')

2026-03-13 11:27:30.614208: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 11:27:30.670185: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-13 11:27:31.904128: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Test images : 379
Modèles disponibles : [PosixPath('/home/onyxia/work/statapp/models/patch_cnn_p64.keras'), PosixPath('/home/onyxia/work/statapp/models/patch_cnn_p48.keras'), PosixPath('/home/onyxia/work/statapp/models/patch_cnn_p32.keras')]


In [3]:
# ── Fonctions utilitaires (reprises de cnn_patch_segmentation.ipynb) ──

def load_img(path, size=IMG_SIZE):
    img = Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

def load_mask(path, size=IMG_SIZE):
    m = Image.open(path).convert('L').resize((size, size), Image.NEAREST)
    return (np.array(m) > 127).astype(np.float32)

def dice(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    denom = np.sum(y_true) + np.sum(y_pred)
    return 2.0 * inter / denom if denom > 0 else 1.0

def iou(y_true, y_pred):
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return inter / union if union > 0 else 1.0

def pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision_score(y_true, y_pred):
    tp = np.sum(y_true * y_pred)
    fp = np.sum((1 - y_true) * y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall_score(y_true, y_pred):
    tp = np.sum(y_true * y_pred)
    fn = np.sum(y_true * (1 - y_pred))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def compute_all_metrics(gt, pred):
    return {
        'dice': dice(gt, pred), 'iou': iou(gt, pred),
        'accuracy': pixel_accuracy(gt, pred),
        'precision': precision_score(gt, pred),
        'recall': recall_score(gt, pred),
    }

def keep_largest_component(mask_bin):
    labeled, n = ndimage_label(mask_bin)
    if n <= 1:
        return mask_bin
    sizes = [np.sum(labeled == k) for k in range(1, n + 1)]
    largest = np.argmax(sizes) + 1
    return (labeled == largest).astype(np.float32)

def predict_segmentation(model, img, patch_size, stride=8, batch_size=256):
    h, w = img.shape[:2]
    half = patch_size // 2
    ys = np.arange(half, h - half, stride)
    xs = np.arange(half, w - half, stride)
    patches = []
    for y in ys:
        for x in xs:
            patches.append(img[y-half:y+half, x-half:x+half])
    patches = np.array(patches, dtype=np.float32)
    preds = model.predict(patches, batch_size=batch_size, verbose=0).flatten()
    grid = preds.reshape(len(ys), len(xs))
    grid_tensor = tf.constant(grid[np.newaxis, ..., np.newaxis], dtype=tf.float32)
    prob_map = tf.image.resize(grid_tensor, (h, w), method='bilinear')[0, :, :, 0].numpy()
    return prob_map

def segment_with_cnn(model, img, patch_size, stride=8, threshold=0.5):
    prob_map = predict_segmentation(model, img, patch_size, stride)
    mask_bin = (prob_map >= threshold).astype(np.float32)
    mask_pp  = keep_largest_component(mask_bin)
    return mask_pp, mask_bin, prob_map

print('Fonctions chargées.')

Fonctions chargées.


---
## 1. K-means LAB sur le test set ISBI 2016 (379 images)

Le K-means est non supervisé, donc on peut l'évaluer directement sur le test set.
On utilise la même configuration que dans la section résultats : LAB, K=2, heuristique sombre, post-traitement LCC.

In [4]:
def kmeans_lab_segment(img_rgb, k=2):
    """
    K-means dans l'espace LAB, heuristique sombre, post-traitement LCC.
    img_rgb : (H, W, 3) float32 [0, 1]
    """
    img_lab = color.rgb2lab(img_rgb)
    h, w, c = img_lab.shape
    pixels = img_lab.reshape(-1, c)
    
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(pixels)
    
    # Heuristique sombre : le cluster avec le L* le plus bas = lésion
    cluster_L = [pixels[labels == i, 0].mean() for i in range(k)]
    lesion_cluster = np.argmin(cluster_L)
    
    mask = (labels == lesion_cluster).reshape(h, w).astype(np.float32)
    mask = keep_largest_component(mask)
    
    return mask

# ── Évaluer sur les 379 images de test ──
test_img_paths  = sorted([str(f) for f in TEST_IMG_DIR.glob('*.jpg')])
test_mask_paths = sorted([str(f) for f in TEST_MASK_DIR.glob('*.png')])
print(f'Test set : {len(test_img_paths)} images, {len(test_mask_paths)} masques')

kmeans_test_results = []
t0 = time.time()

for i, (ip, mp) in enumerate(zip(test_img_paths, test_mask_paths)):
    img = load_img(ip)
    gt  = load_mask(mp)
    rho = float(np.mean(gt))
    
    pred = kmeans_lab_segment(img, k=2)
    metrics = compute_all_metrics(gt, pred)
    metrics['image'] = Path(ip).stem
    metrics['rho'] = rho
    kmeans_test_results.append(metrics)
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(test_img_paths) - i - 1)
        print(f'  {i+1}/{len(test_img_paths)} ({elapsed:.0f}s, ETA ~{eta:.0f}s)')

elapsed = time.time() - t0
df_km = pd.DataFrame(kmeans_test_results)
print(f'\nK-means LAB sur test set ({len(df_km)} images) en {elapsed:.0f}s')

Test set : 379 images, 379 masques
  50/379 (8s, ETA ~54s)
  100/379 (16s, ETA ~46s)
  150/379 (26s, ETA ~40s)
  200/379 (37s, ETA ~33s)
  250/379 (45s, ETA ~23s)
  300/379 (53s, ETA ~14s)
  350/379 (61s, ETA ~5s)

K-means LAB sur test set (379 images) en 66s


In [5]:
# ── Résultats globaux K-means test ──
print('=' * 60)
print('K-MEANS LAB — TEST SET ISBI 2016')
print('=' * 60)
for col in ['dice', 'iou', 'accuracy', 'precision', 'recall']:
    print(f'  {col.upper():<12} {df_km[col].mean():.4f} ± {df_km[col].std():.4f}')
n_echecs = (df_km['dice'] < 0.5).sum()
print(f'  Échecs (DSC<0.5) : {n_echecs} ({100*n_echecs/len(df_km):.1f}%)')

# ── Par sous-groupes de taille ──
print(f'\n{"─" * 60}')
print('Par sous-groupe de taille :')
for label, lo, hi in [('Petites (ρ<5%)', 0, 0.05), 
                       ('Moyennes (5-10%)', 0.05, 0.10), 
                       ('Grandes (≥10%)', 0.10, 1.01)]:
    sub = df_km[(df_km['rho'] >= lo) & (df_km['rho'] < hi)]
    if len(sub) > 0:
        sub_echecs = (sub['dice'] < 0.5).sum()
        print(f'  {label:<22} n={len(sub):>3}  '
              f'DSC={sub["dice"].mean():.3f}±{sub["dice"].std():.3f}  '
              f'échecs={sub_echecs}')

print(f'\n{"─" * 60}')
print('VALEURS À REPORTER DANS LE LATEX :')
print(f'  DSC global test : {df_km["dice"].mean():.3f}')
print(f'  Taux échec test : {100*n_echecs/len(df_km):.1f}%')
for label, lo, hi in [('Petites', 0, 0.05), ('Moyennes', 0.05, 0.10), ('Grandes', 0.10, 1.01)]:
    sub = df_km[(df_km['rho'] >= lo) & (df_km['rho'] < hi)]
    if len(sub) > 0:
        print(f'  DSC {label} : {sub["dice"].mean():.3f}')

K-MEANS LAB — TEST SET ISBI 2016
  DICE         0.6483 ± 0.3460
  IOU          0.5612 ± 0.3230
  ACCURACY     0.8396 ± 0.1828
  PRECISION    0.7722 ± 0.4027
  RECALL       0.6109 ± 0.3183
  Échecs (DSC<0.5) : 90 (23.7%)

────────────────────────────────────────────────────────────
Par sous-groupe de taille :
  Petites (ρ<5%)         n= 40  DSC=0.197±0.358  échecs=32
  Moyennes (5-10%)       n= 47  DSC=0.411±0.417  échecs=25
  Grandes (≥10%)         n=292  DSC=0.748±0.253  échecs=33

────────────────────────────────────────────────────────────
VALEURS À REPORTER DANS LE LATEX :
  DSC global test : 0.648
  Taux échec test : 23.7%
  DSC Petites : 0.197
  DSC Moyennes : 0.411
  DSC Grandes : 0.748


---
## 2. Précision/Rappel pour chaque taille de patch (P=32, 48, 64)

Le tableau d'ablation dans le mémoire a besoin des colonnes Précision et Rappel pour les 3 tailles.
On charge chaque modèle et on évalue sur la validation (180 images).

In [6]:
# ── Reconstruire le split train/val identique ──
all_img_paths  = sorted([str(f) for f in IMAGES_DIR.glob('*.jpg')])
all_mask_paths = sorted([str(f) for f in MASKS_DIR.glob('*.png')])

# Calculer rho pour le split stratifié
rhos = []
for mp in all_mask_paths:
    m = np.array(Image.open(mp).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
    rhos.append(np.sum(m > 127) / (IMG_SIZE * IMG_SIZE))
rhos = np.array(rhos)

# Même stratification que dans le notebook original
rho_bins = np.digitize(rhos, bins=[0.01, 0.05, 0.10, 0.20, 0.40])

# Split : d'abord séparer test (10%), puis train/val sur le reste
all_indices = np.arange(len(all_img_paths))
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
trainval_idx, test_idx = next(sss1.split(all_indices, rho_bins))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2/0.9, random_state=SEED)
train_sub, val_sub = next(sss2.split(trainval_idx, rho_bins[trainval_idx]))
train_idx = trainval_idx[train_sub]
val_idx   = trainval_idx[val_sub]

val_img  = [all_img_paths[i] for i in val_idx]
val_mask = [all_mask_paths[i] for i in val_idx]

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

Train: 630 | Val: 180 | Test: 90


In [7]:
# ── Évaluer chaque taille de patch ──
PATCH_SIZES = [32, 48, 64]
STRIDES = {32: 8, 48: 8, 64: 10}

ablation_full = {}

for ps in PATCH_SIZES:
    print(f'\n{"="*60}')
    print(f'  P = {ps}, stride = {STRIDES[ps]}')
    print(f'{"="*60}')
    
    model = tf.keras.models.load_model(SAVE_DIR / f'patch_cnn_p{ps}.keras')
    stride = STRIDES[ps]
    
    results_pp = []   # avec post-traitement
    results_raw = []  # sans post-traitement
    
    t0 = time.time()
    for i, (ip, mp) in enumerate(zip(val_img, val_mask)):
        img = load_img(ip)
        gt  = load_mask(mp)
        
        prob_map = predict_segmentation(model, img, ps, stride)
        mask_raw = (prob_map >= 0.5).astype(np.float32)
        mask_pp  = keep_largest_component(mask_raw)
        
        results_raw.append(compute_all_metrics(gt, mask_raw))
        results_pp.append(compute_all_metrics(gt, mask_pp))
        
        if (i + 1) % 30 == 0:
            elapsed = time.time() - t0
            print(f'  {i+1}/{len(val_img)} ({elapsed:.0f}s)')
    
    df_pp  = pd.DataFrame(results_pp)
    df_raw = pd.DataFrame(results_raw)
    
    ablation_full[ps] = {'pp': df_pp, 'raw': df_raw}
    
    # Résultats avec PP (pour le tableau d'ablation)
    print(f'\n  Avec post-traitement :')
    for col in ['dice', 'iou', 'precision', 'recall']:
        print(f'    {col.upper():<12} {df_pp[col].mean():.4f} ± {df_pp[col].std():.4f}')
    print(f'    Échecs : {(df_pp["dice"] < 0.5).sum()}')

print('\nTerminé.')


  P = 32, stride = 8


I0000 00:00:1773401327.758046  305316 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1226 MB memory:  -> device: 0, name: NVIDIA A2, pci bus id: 0000:18:00.0, compute capability: 8.6
2026-03-13 11:28:49.015740: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f3358014540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-13 11:28:49.015784: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A2, Compute Capability 8.6
2026-03-13 11:28:49.030551: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-13 11:28:49.086052: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90501
2026-03-13 11:28:49.105691: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none 

  30/180 (9s)
  60/180 (15s)
  90/180 (21s)
  120/180 (27s)
  150/180 (33s)
  180/180 (39s)

  Avec post-traitement :
    DICE         0.7651 ± 0.1512
    IOU          0.6386 ± 0.1607
    PRECISION    0.6854 ± 0.1696
    RECALL       0.9305 ± 0.1745
    Échecs : 8

  P = 48, stride = 8


2026-03-13 11:29:30.984107: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


  30/180 (11s)
  60/180 (18s)
  90/180 (25s)
  120/180 (33s)
  150/180 (41s)
  180/180 (49s)

  Avec post-traitement :
    DICE         0.7323 ± 0.1530
    IOU          0.5960 ± 0.1563
    PRECISION    0.6341 ± 0.1700
    RECALL       0.9302 ± 0.1795
    Échecs : 10

  P = 64, stride = 10


2026-03-13 11:30:21.143268: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.


  30/180 (14s)
  60/180 (22s)
  90/180 (29s)
  120/180 (36s)
  150/180 (42s)
  180/180 (49s)

  Avec post-traitement :
    DICE         0.7073 ± 0.1377
    IOU          0.5621 ± 0.1441
    PRECISION    0.6027 ± 0.1658
    RECALL       0.9328 ± 0.1715
    Échecs : 9

Terminé.


In [8]:
# ══════════════════════════════════════════════════════════════
# VALEURS À REPORTER DANS LE LATEX
# ══════════════════════════════════════════════════════════════

print('=' * 70)
print('TABLEAU D\'ABLATION — Précision et Rappel à ajouter')
print('=' * 70)
print(f'{"P":<6} {"Précision":>12} {"Rappel":>12}')
print('-' * 30)
for ps in [32, 48, 64]:
    df = ablation_full[ps]['pp']
    print(f'{ps:<6} {df["precision"].mean():.3f}         {df["recall"].mean():.3f}')

print(f'\n{"="*70}')
print('TABLEAU POST-TRAITEMENT P=32 (sans PP vs avec PP)')
print('=' * 70)
df_raw = ablation_full[32]['raw']
df_pp  = ablation_full[32]['pp']

print(f'{"Métrique":<12} {"Sans PP":>20} {"Avec PP":>20} {"Δ":>8}')
print('-' * 62)
for col in ['dice', 'iou', 'accuracy', 'precision', 'recall']:
    m1 = df_raw[col].mean()
    s1 = df_raw[col].std()
    m2 = df_pp[col].mean()
    s2 = df_pp[col].std()
    delta = m2 - m1
    print(f'{col.upper():<12} {m1:.3f} ± {s1:.3f}      {m2:.3f} ± {s2:.3f}      {delta:+.3f}')

n_fail_raw = (df_raw['dice'] < 0.5).sum()
n_fail_pp  = (df_pp['dice'] < 0.5).sum()
print(f'{"Échecs":<12} {n_fail_raw:>20} {n_fail_pp:>20} {n_fail_pp - n_fail_raw:>+8}')

TABLEAU D'ABLATION — Précision et Rappel à ajouter
P         Précision       Rappel
------------------------------
32     0.685         0.930
48     0.634         0.930
64     0.603         0.933

TABLEAU POST-TRAITEMENT P=32 (sans PP vs avec PP)
Métrique                  Sans PP              Avec PP        Δ
--------------------------------------------------------------
DICE         0.751 ± 0.142      0.765 ± 0.151      +0.014
IOU          0.619 ± 0.163      0.639 ± 0.161      +0.019
ACCURACY     0.867 ± 0.109      0.875 ± 0.117      +0.008
PRECISION    0.662 ± 0.180      0.685 ± 0.170      +0.024
RECALL       0.942 ± 0.132      0.930 ± 0.174      -0.012
Échecs                         14                    8       -6


In [9]:
# ══════════════════════════════════════════════════════════════
# RÉSUMÉ COMPLET — copier-coller pour mettre à jour le LaTeX
# ══════════════════════════════════════════════════════════════

print('\n' + '█' * 70)
print('  RÉCAPITULATIF DES VALEURS À REPORTER')
print('█' * 70)

print('\n── 1. K-means LAB test set ──')
print(f'  DSC global : {df_km["dice"].mean():.3f}')
n_echecs_km = (df_km['dice'] < 0.5).sum()
print(f'  Taux échec : {100*n_echecs_km/len(df_km):.1f}%')
for label, lo, hi in [('Petites', 0, 0.05), ('Moyennes', 0.05, 0.10), ('Grandes', 0.10, 1.01)]:
    sub = df_km[(df_km['rho'] >= lo) & (df_km['rho'] < hi)]
    if len(sub) > 0:
        print(f'  DSC {label} (n={len(sub)}) : {sub["dice"].mean():.3f}')

print('\n── 2. Tableau ablation : Précision / Rappel ──')
for ps in [32, 48, 64]:
    df = ablation_full[ps]['pp']
    print(f'  P={ps} : Précision={df["precision"].mean():.3f}, Rappel={df["recall"].mean():.3f}')

print('\n── 3. Tableau post-traitement P=32 ──')
for label, df in [('Sans PP', ablation_full[32]['raw']), ('Avec PP', ablation_full[32]['pp'])]:
    print(f'  {label} : DSC={df["dice"].mean():.3f}±{df["dice"].std():.3f}, '
          f'IoU={df["iou"].mean():.3f}±{df["iou"].std():.3f}, '
          f'Acc={df["accuracy"].mean():.3f}±{df["accuracy"].std():.3f}, '
          f'Prec={df["precision"].mean():.3f}±{df["precision"].std():.3f}, '
          f'Recall={df["recall"].mean():.3f}±{df["recall"].std():.3f}, '
          f'Échecs={(df["dice"] < 0.5).sum()}')

print('\n' + '█' * 70)
print('  Envoie-moi ces résultats et je mets à jour le LaTeX !')
print('█' * 70)


██████████████████████████████████████████████████████████████████████
  RÉCAPITULATIF DES VALEURS À REPORTER
██████████████████████████████████████████████████████████████████████

── 1. K-means LAB test set ──
  DSC global : 0.648
  Taux échec : 23.7%
  DSC Petites (n=40) : 0.197
  DSC Moyennes (n=47) : 0.411
  DSC Grandes (n=292) : 0.748

── 2. Tableau ablation : Précision / Rappel ──
  P=32 : Précision=0.685, Rappel=0.930
  P=48 : Précision=0.634, Rappel=0.930
  P=64 : Précision=0.603, Rappel=0.933

── 3. Tableau post-traitement P=32 ──
  Sans PP : DSC=0.751±0.142, IoU=0.619±0.163, Acc=0.867±0.109, Prec=0.662±0.180, Recall=0.942±0.132, Échecs=14
  Avec PP : DSC=0.765±0.151, IoU=0.639±0.161, Acc=0.875±0.117, Prec=0.685±0.170, Recall=0.930±0.174, Échecs=8

██████████████████████████████████████████████████████████████████████
  Envoie-moi ces résultats et je mets à jour le LaTeX !
██████████████████████████████████████████████████████████████████████
